# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets and their field `@id`s.

In [ ]:
# List all available record sets and display their fields and columns
print("Available Record Sets (@id):")
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id} | name: {getattr(rs, 'name', None)}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (@id):")
        for field in rs.fields:
            print(f"    - @id: {field.id}, name: {getattr(field, 'name', None)} (type: {getattr(field, 'data_type', None)})")
            if hasattr(field, 'source') and field.source and hasattr(field.source, 'column'):
                col = field.source.column
                print(f"      -> column: {col.id if hasattr(col,'id') else col}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all available record sets. If you wish to use a specific one, please select the `@id` from above.

In [ ]:
# Load all dataframes from each record set
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"  Fields: {df.columns.tolist()}")
        print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Typical operations include removing outliers, transforming data distributions, or grouping by key attributes for further analysis.

***Please adjust the code below to select a specific record set and fields based on the data overview above.***

In [ ]:
# Set the record set @id you want to analyze
# Replace with the actual record set you wish to analyze (from the overview section)
selected_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[selected_record_set_id] if selected_record_set_id else pd.DataFrame()

# Show the numeric columns
numeric_columns = df.select_dtypes(include=["number"]).columns.tolist()
print("Numeric columns:", numeric_columns)

if numeric_columns:
    numeric_field = numeric_columns[0]  # You may replace this as needed (e.g. 'MW' for molecular weight)
    threshold = df[numeric_field].mean() if not df[numeric_field].empty else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
    print(filtered_df.head())
    
    # Normalize the field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}':")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by a likely categorical field, e.g. 'description' if present
    possible_group_fields = [col for col in df.columns if col.lower() in ('description','group','accession')]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
        print(grouped_df.head())
else:
    print("No numeric columns found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For demonstration, we create a histogram for the selected numeric field and a barplot of its mean by a group (if such data exists).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we found a group_field above, plot means by group
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(12,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric data available to plot.")

## 6. Conclusion
In this notebook, we:
- Loaded dataset metadata and record sets using `mlcroissant`
- Explored the structure and data fields of the record sets (referenced by their `@id`)
- Extracted data into DataFrames for analysis
- Applied basic filtering, normalization, and aggregation for exploratory analysis
- Visualized data distributions and, where possible, relationships by grouping attributes

### Next steps
- Refine the attribute selection (fields and groupings) for domain-specific analysis
- Explore additional record sets or fields if available
- Apply further statistical or machine learning methods relevant to your research question
